# 185. Department Top Three Salaries
https://leetcode.com/problems/department-top-three-salaries/

## Complexity Comparison

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| DENSE_RANK() Window | O(n log n) | O(n) | Cleanest, recommended |
| Correlated Subquery COUNT | O(n^2) | O(1) | Count distinct higher salaries |
| Self-Join with DISTINCT | O(n^2) | O(n) | Older SQL style |

## Methodology

This is a SQL problem (Hard). Find employees whose salary is in the top 3 unique
salaries within their department. We use DENSE_RANK() partitioned by department,
ordered by salary descending, then filter for rank <= 3. DENSE_RANK handles ties
correctly: if two people share the highest salary, both get rank 1.

## Solutions

### C#

In [ ]:
// SQL Problem - Department Top Three Salaries
// SQL Solution:
//
// SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
// FROM (
//     SELECT name, salary, departmentId,
//            DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) AS rnk
//     FROM Employee
// ) e
// JOIN Department d ON e.departmentId = d.id
// WHERE e.rnk <= 3;
//
// Alternative using correlated subquery:
//
// SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
// FROM Employee e
// JOIN Department d ON e.departmentId = d.id
// WHERE (
//     SELECT COUNT(DISTINCT e2.salary)
//     FROM Employee e2
//     WHERE e2.departmentId = e.departmentId AND e2.salary > e.salary
// ) < 3;

// C# programmatic equivalent
using System;
using System.Linq;
using System.Collections.Generic;

public class Employee { public string Name; public int Salary; public int DeptId; }
public class Department { public int Id; public string Name; }

public class Solution
{
    public static List<(string dept, string emp, int salary)> DeptTopThree(
        List<Employee> employees, List<Department> departments)
    {
        var deptMap = departments.ToDictionary(d => d.Id, d => d.Name);
        var topSalaries = employees.GroupBy(e => e.DeptId)
            .ToDictionary(g => g.Key,
                g => g.Select(e => e.Salary).Distinct()
                      .OrderByDescending(s => s).Take(3).ToHashSet());
        return employees
            .Where(e => topSalaries.ContainsKey(e.DeptId)
                     && topSalaries[e.DeptId].Contains(e.Salary))
            .Select(e => (deptMap[e.DeptId], e.Name, e.Salary))
            .ToList();
    }
}

// Test
var emps = new List<Employee>
{
    new Employee { Name = "Joe", Salary = 85000, DeptId = 1 },
    new Employee { Name = "Henry", Salary = 80000, DeptId = 2 },
    new Employee { Name = "Sam", Salary = 60000, DeptId = 2 },
    new Employee { Name = "Max", Salary = 90000, DeptId = 1 },
    new Employee { Name = "Janet", Salary = 69000, DeptId = 1 },
    new Employee { Name = "Randy", Salary = 85000, DeptId = 1 },
    new Employee { Name = "Will", Salary = 70000, DeptId = 1 }
};
var depts = new List<Department>
{
    new Department { Id = 1, Name = "IT" },
    new Department { Id = 2, Name = "Sales" }
};
foreach (var r in Solution.DeptTopThree(emps, depts))
    Console.WriteLine($"{r.dept}: {r.emp} ({r.salary})");

### Python

In [ ]:
// SQL Problem - Python programmatic equivalent
//
// from collections import defaultdict
//
// def dept_top_three(employees, departments):
//     dept_map = {d['id']: d['name'] for d in departments}
//     by_dept = defaultdict(list)
//     for e in employees:
//         by_dept[e['deptId']].append(e['salary'])
//     top3 = {}
//     for dept_id, sals in by_dept.items():
//         top3[dept_id] = set(sorted(set(sals), reverse=True)[:3])
//     return [
//         (dept_map[e['deptId']], e['name'], e['salary'])
//         for e in employees
//         if e['salary'] in top3.get(e['deptId'], set())
//     ]

### Go

In [ ]:
// SQL Problem - Go programmatic equivalent
//
// func deptTopThree(employees []Employee, departments []Department) []Result {
//     deptMap := make(map[int]string)
//     for _, d := range departments { deptMap[d.ID] = d.Name }
//     byDept := make(map[int]map[int]bool)
//     for _, e := range employees {
//         if byDept[e.DeptID] == nil { byDept[e.DeptID] = make(map[int]bool) }
//         byDept[e.DeptID][e.Salary] = true
//     }
//     top3 := make(map[int]map[int]bool)
//     for deptID, sals := range byDept {
//         sorted := make([]int, 0, len(sals))
//         for s := range sals { sorted = append(sorted, s) }
//         sort.Sort(sort.Reverse(sort.IntSlice(sorted)))
//         top3[deptID] = make(map[int]bool)
//         for i := 0; i < len(sorted) && i < 3; i++ {
//             top3[deptID][sorted[i]] = true
//         }
//     }
//     var result []Result
//     for _, e := range employees {
//         if top3[e.DeptID][e.Salary] {
//             result = append(result, Result{deptMap[e.DeptID], e.Name, e.Salary})
//         }
//     }
//     return result
// }

### Rust

In [ ]:
// SQL Problem - Rust programmatic equivalent
//
// use std::collections::{HashMap, BTreeSet, HashSet};
//
// fn dept_top_three(
//     employees: &[Employee],
//     departments: &[Department],
// ) -> Vec<(String, String, i32)> {
//     let dept_map: HashMap<i32, &str> =
//         departments.iter().map(|d| (d.id, d.name.as_str())).collect();
//     let mut by_dept: HashMap<i32, BTreeSet<i32>> = HashMap::new();
//     for e in employees {
//         by_dept.entry(e.dept_id).or_default().insert(e.salary);
//     }
//     let top3: HashMap<i32, HashSet<i32>> = by_dept.into_iter().map(|(dept, sals)| {
//         let top: HashSet<i32> = sals.into_iter().rev().take(3).collect();
//         (dept, top)
//     }).collect();
//     employees.iter()
//         .filter(|e| top3.get(&e.dept_id).map_or(false, |t| t.contains(&e.salary)))
//         .map(|e| (dept_map[&e.dept_id].to_string(), e.name.clone(), e.salary))
//         .collect()
// }

## Example Scenarios

1. **Normal top 3**: IT dept salaries [90k, 85k, 85k, 70k, 69k] -> top 3 unique: 90k, 85k, 70k (4 employees)
2. **Fewer than 3 distinct**: Sales has only [80k, 60k] -> both returned
3. **All same salary**: Everyone in dept earns 50k -> all returned (rank 1 for all)
4. **Ties at rank boundary**: 3rd and 4th highest are tied -> both included (dense rank)
5. **Single employee**: Department with one person -> that person returned

*Infographic will be added in a future update.*